In [25]:
import os 
import pandas as pd
import chardet
from chardet.universaldetector import UniversalDetector  

# 获取当前文件绝对路径
path = os.path.abspath('.')
# source_path
sp = os.path.join(path,'运单源')
sp1 = os.path.join(path,'订单源')
sp2 = os.path.join(path,'过审表源')
sp3 = os.path.join(path,'间接订单源')
sp4 = os.path.join(path,'日志源')
sp5 = os.path.join(path,'售后单源')
sp6 = os.path.join(path,'视频号订单')
sp7 = os.path.join(path,'拼多多订单源')
sp8 = os.path.join(path,'快手订单源')

In [26]:
def get_encoding(file):
    # 二进制方式读取，获取字节数据，检测类型
    with open(file, 'rb') as f:
        return chardet.detect(f.read())

In [27]:
# 检测编码
detector = UniversalDetector()

for root, dirs, files in os.walk(sp1):
    for file in files:
        temp = os.path.join(sp1,file)
        with open(temp,'rb') as f:
            for line in f.readlines():
                detector.feed(line)
        print(detector.result)

#关闭检测
detector.close()

{'encoding': None, 'confidence': 0.0, 'language': None}

In [28]:
frames_dd = []
frames_yd = []
frames_jj = []
frames_log = []
frames_sh = []
frames_sph = []
frames_ks = []
frames_pdd =[]
# 订单
for root, dirs, files in os.walk(sp1):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp,converters={'商品ID':str},engine='python',encoding='utf-8-sig') # 订单：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp) # 订单：（xls、xlsx）->dataframe
        file_name = os.path.splitext(os.path.basename(file))[0]
        df.insert(0, '店铺名称', file_name)
        frames_dd.append(df)
        
# 拼多多订单
for root, dirs, files in os.walk(sp7):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp,converters={'商品ID':str},engine='python',encoding='utf-8-sig') # 订单：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp) # 订单：（xls、xlsx）->dataframe
        file_name = os.path.splitext(os.path.basename(file))[0]
        df.insert(0, '店铺名称', file_name)
        frames_pdd.append(df)

        
# 运单
for root, dirs, files in os.walk(sp):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp) # 运单：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp,converters={'订单编号':str}) # 运单：（xls、xlsx）->dataframe
        frames_yd.append(df)
#售后单
for root, dirs, files in os.walk(sp5):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp) # 运单：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp,converters={'订单号':str}) # 运单：（xls、xlsx）->dataframe
        frames_sh.append(df)
#视频号
for root, dirs, files in os.walk(sp6):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp) # 运单：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp,converters={'订单号':str}) # 运单：（xls、xlsx）->dataframe
        file_name = os.path.splitext(os.path.basename(file))[0]
        df.insert(0, '店铺名称', file_name)
        frames_sph.append(df)
#快手
for root, dirs, files in os.walk(sp8):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp) # 运单：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp,converters={'订单号':str}) # 运单：（xls、xlsx）->dataframe
        file_name = os.path.splitext(os.path.basename(file))[0]
        df.insert(0, '店铺名称', file_name)
        frames_ks.append(df)
# 间接订单
for root, dirs, files in os.walk(sp3):
    for file in files:
        fp = os.path.join(root,file)
        if os.path.splitext(file)[1] == '.csv':
            df = pd.read_csv(fp) # 间接：csv->dataframe
        if os.path.splitext(file)[1] == '.xlsx' or os.path.splitext(file)[1] == '.xls':
            df = pd.read_excel(fp) # 间接：（xls、xlsx）->dataframe
        frames_jj.append(df)

# 获取文件列表
dirList = os.listdir(sp2)
# 遍历拓展名为.xlsx的文件
df_list = []
for i in dirList:
    temp = os.path.splitext(i)
    if temp[0] != '过审状况表' and temp[1] == '.xlsx':
        sp3 = os.path.join(sp2,i)
        df = pd.read_excel(sp3)
        df_list.append(df)
        
# 日志
for root, dirs, files in os.walk(sp4):
    for file in files:
        fp = os.path.join(root,file)
        df = pd.read_csv(fp) # 日志：csv->dataframe 
        frames_log.append(df)

In [29]:
#写入1.csv
if frames_dd:
    df_dd = pd.concat(frames_dd)
    df_dd["主订单编号"] = df_dd["主订单编号"].astype(str)+'\t'# 超过15位的数字需要转str
    df_dd["子订单编号"] = df_dd["子订单编号"].astype(str)+'\t'
    df_dd["达人ID"] = df_dd["达人ID"].astype(str)+'\t'
    new_dd_name = os.path.join(path,'1.csv')
    df_dd.to_csv(new_dd_name, index=False, encoding='utf-8-sig')

In [30]:
#写入过审状况表
if dirList:
    df1 = pd.concat(df_list)
    new_file_name = os.path.join(path,'过审状况表.xlsx')
    df1.to_excel(new_file_name,index=False,encoding='gbk')

In [31]:
#写入间接订单表中
if frames_jj:
    df_jj = pd.concat(frames_jj)
    df_jj['订单编号'] = df_jj['订单编号'].astype(str)+ '\t' # 超过15位的数字需要转str
    new_jj_name = os.path.join(path,'间接订单表.csv')
    df_jj.to_csv(new_jj_name, index=False, encoding='utf-8')

In [32]:
#写入日志表中
if frames_log:
    df_log = pd.concat(frames_log)
    newFileName = os.path.join(path,'日志.xlsx')
    df_log.to_excel(newFileName,index=False,encoding='utf-8-sig')

In [33]:
#写入售后表
if frames_sh:
    df_sh = pd.concat(frames_sh)
    df_sh['订单号'] = df_sh['订单号'].astype(str)+ '\t' # 超过15位的数字需要转str
    #new_yd_name = os.path.join(path,'2.csv')
    #df_yd.to_csv(new_yd_name, index=False, encoding='utf-8-sig')
    new_sh_name = os.path.join(path,'售后.xlsx')
    df_sh.to_excel(new_sh_name, index=False, encoding='utf-8-sig')

In [34]:
#写入视频号表
if frames_sph:
    df_sph = pd.concat(frames_sph)
    df_sph['订单号'] = df_sph['订单号'].astype(str)+ '\t' # 超过15位的数字需要转str
    df_sph['商品编码(平台)'] = df_sph['商品编码(平台)'].astype(str)+ '\t'
    #new_yd_name = os.path.join(path,'2.csv')
    #df_yd.to_csv(new_yd_name, index=False, encoding='utf-8-sig')
    new_sph_name = os.path.join(path,'视频号.xlsx')
    df_sph.to_excel(new_sph_name, index=False, encoding='utf-8-sig')

In [35]:
#写入快手表
if frames_ks:
    df_ks = pd.concat(frames_ks)
    df_ks['订单号'] = df_ks['订单号'].astype(str)+ '\t' # 超过15位的数字需要转str
    df_ks['SKU编码'] = df_ks['SKU编码'].astype(str)+ '\t'
    #new_yd_name = os.path.join(path,'2.csv')
    #df_yd.to_csv(new_yd_name, index=False, encoding='utf-8-sig')
    new_ks_name = os.path.join(path,'快手.xlsx')
    df_ks.to_excel(new_ks_name, index=False, encoding='utf-8-sig')

In [36]:
#写入pdd.csv
if frames_pdd:
    df_pdd = pd.concat(frames_pdd)
    df_pdd["订单号"] = df_pdd["订单号"].astype(str)+'\t'# 超过15位的数字需要转str
    df_pdd["快递单号"] = df_pdd["快递单号"].astype(str)+'\t'
    new_pdd_name = os.path.join(path,'pdd.csv')
    df_pdd.to_csv(new_pdd_name, index=False, encoding='utf-8-sig')